# Natural Language Processing (NLP) for Data Science — Executable Code Notebook
**Classroom Reference & Runnable Examples**

This notebook extracts all code snippets along with their module/section titles, step-by-step explanations, detailed breakdowns of Regular Expression tokens (`\w`, `\s`, `\d`, `\S`, `\b`, `^`, `*?`, `\1`, etc.), and the complete end-to-end Customer Complaint Classification project.

> **Note:** Run the `nltk.download(...)` lines once to download required tokenizers, stop words, lemmatizers, POS taggers, and sentiment lexicons before running NLTK cells.

---
## MODULE 2: UNDERSTANDING TEXT DATA
### 2.5 Why Text Cannot Directly Be Passed into Traditional ML
Traditional ML algorithms (such as `LogisticRegression`) expect numerical matrices rather than raw strings. Passing raw strings directly into `.fit()` raises a `ValueError`.

In [2]:
from sklearn.linear_model import LogisticRegression

X = ["I love Python", "I hate Python"]
y = [1, 0]

model = LogisticRegression()

# This will not work directly because X contains raw strings instead of numbers:
# model.fit(X, y)  # Raises ValueError: could not convert string to float

print("Raw text X:", X)
print("Target y:", y)
print("Text must first be vectorized (e.g., CountVectorizer or TfidfVectorizer) before calling model.fit().")

Raw text X: ['I love Python', 'I hate Python']
Target y: [1, 0]
Text must first be vectorized (e.g., CountVectorizer or TfidfVectorizer) before calling model.fit().


---
## MODULE 3: NLP TEXT PREPROCESSING
### 3.2 Lowercasing
Converts uppercase characters to lowercase so words like `"Fraud"`, `"FRAUD"`, and `"fraud"` are treated identically.

In [2]:
text = "DATA Science Is INTERESTING"
clean_text = text.lower()
print(clean_text)

data science is interesting


### 3.3 Removing Extra Spaces
* `text.split()` splits the string on any whitespace and discards leading, trailing, and consecutive spaces.
* `" ".join(...)` rejoins the words with a single space.

In [6]:
text = "  I   love    NLP   "
clean_text = " ".join(text.split())
print(clean_text)

I love NLP


### 3.4 Removing Punctuation
**Pattern Explanation (`r"[^\w\s]"`):**
* `r"..."`: Raw string literal so backslashes are passed directly to the regex engine.
* `[...]`: Character set matching any single character inside the brackets.
* `^` (inside `[...]`): Negation — matches anything **NOT** in the set.
* `\w`: Matches any **word character** (letters `a-z`, `A-Z`, digits `0-9`, and underscore `_`).
* `\s`: Matches any **whitespace character** (spaces, tabs, newlines).
* Combined: `[^\w\s]` matches any character that is neither a word character nor whitespace (i.e., punctuation/symbols) and replaces it with `""`.

In [7]:
import re

text = "Hello!!! How are you?"
clean_text = re.sub(r"[^\w\s]", "", text)
print(clean_text)

Hello How are you


In [2]:
import re

text = "Hello!!! How are you?"
clean_text = re.sub(r"[^\w\s]", "|", text)
print(clean_text)

Hello||| How are you|


### 3.5 Removing Special Characters
**Pattern Explanation (`r"[^a-zA-Z0-9\s]"`):**
* `[^...]`: Negated character set (matches anything **not** listed).
* `a-z`: Lowercase English letters.
* `A-Z`: Uppercase English letters.
* `0-9`: Digits.
* `\s`: Whitespace characters.
* Unlike `\w`, this explicitly excludes underscores `_` and non-ASCII symbols, keeping only alphanumeric characters and spaces.

In [25]:
text = "Payment @#$ failed!!!"
clean_text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
print(clean_text)

Payment  failed


### 3.6 Removing HTML Tags
**Pattern Explanation (`r"<.*?>"`):**
* `<` and `>`: Literal opening and closing angle brackets of an HTML tag.
* `.`: Matches any character except a newline.
* `*?`: **Non-greedy** zero-or-more quantifier — stops at the **first** closing `>` rather than matching across multiple tags.

In [6]:
text = "<p>Payment failed</p>"
clean_text = re.sub(r"<.*?>", "", text)
print(clean_text)

Payment failed


### 3.7 Removing URLs
**Pattern Explanation (`r"https?://\S+|www\.\S+"`):**
* `https?://`: Matches `http://` or `https://` (`s?` makes the `s` optional — zero or one occurrence).
* `\S+`: Matches one or more **non-whitespace characters** (continues until the next space).
* `|`: Logical **OR** operator.
* `www\.`: Matches literal `www.` (`\.` escapes the dot so it means a literal period rather than "any character").

In [3]:
t = " this is the "
t.strip()

'this is the'

In [4]:
t.lstrip()

'this is the '

In [7]:
text = "Visit https://example.com for details"
clean_text = re.sub(r"https?://\S+|www\.\S+", "", text)
print(clean_text.strip())

Visit  for details


### 3.8 Removing Email Addresses
**Pattern Explanation (`r"\S+@\S+"`):**
* `\S+`: One or more non-whitespace characters (username part before `@`).
* `@`: Literal `@` symbol.
* `\S+`: One or more non-whitespace characters (domain part after `@`).

In [8]:
text = "Contact me at user@example.com"
clean_text = re.sub(r"\S+@\S+", "", text)
print(clean_text.strip())

Contact me at


### 3.9 Removing Numbers
**Pattern Explanation (`r"\d+"`):**
* `\d`: Matches any **digit** (`0–9`).
* `+`: Quantifier matching **one or more** consecutive digits (e.g., `500` or `123` as a whole sequence).

In [9]:
text = "I paid 500 rupees for order 123"
clean_text = re.sub(r"\d+", "", text)
print(clean_text)

I paid  rupees for order 


In [5]:
text = "I paid 500 rupees for order 123"
clean_text = re.sub(r"\d+", "||", text)
print(clean_text)

I paid || rupees for order ||


### 3.10 Handling Emojis
Replaces selected sentiment-bearing emojis with descriptive text tokens so emotional context is preserved.

In [10]:
text = "Great service 😊"
text = text.replace("😊", " happy ")
print(text)

Great service  happy 


### 3.11 Handling Contractions
Expands shortened word forms (e.g., `"don't"` → `"do not"`) to preserve negation and semantic clarity.

In [11]:
text = "I don't like this service"
text = text.replace("don't", "do not")
print(text)

I do not like this service


### 3.13 Handling Repeated Characters
**Pattern Explanation (`r"(.)\1{2,}"` replaced with `r"\1"`):**
* `(.)`: Capturing group 1 matching any single character.
* `\1`: Backreference matching the **exact same character** captured in group 1.
* `{2,}`: Repetition quantifier meaning the backreference must repeat **2 or more additional times** (so 3 or more total occurrences, like `ooooo`).
* Replacement `r"\1"`: Replaces the entire repeated sequence with a single copy of that character.

In [40]:
text = "The service was sooooo good"
clean_text = re.sub(r"(.)\1{2,}", r"\1", text)
print(clean_text)

The service was so good


### 3.14 Handling Missing Text & Creating a Missing-Text Indicator
Replaces `None`/`NaN` with an empty string `""` (preventing literal `"nan"` strings) and optionally creates a binary flag column (`1` if missing, `0` otherwise).

In [7]:
import pandas as pd

df = pd.DataFrame({
    "message": ["Payment failed", None, "Card blocked"]
})
df["message"] = df["message"].fillna("")
df["message_missing"] = df["message"].eq("").astype(int)
df

,message,message_missing
0,Payment failed,0
1,,1
2,Card blocked,0


### 3.15 Handling Duplicate Text Records
Removes exact duplicate rows using `.drop_duplicates()` to prevent data distortion and train-test leakage.

In [14]:
df_dedup = df.drop_duplicates()
print(df_dedup)

          message  message_missing
0  Payment failed                0
1                                1
2    Card blocked                0


---
## MODULE 4: REGULAR EXPRESSIONS (REGEX)
### 4.1 Extracting Numbers with `re.findall()`
* `\d+`: Matches one or more consecutive digits (`0–9`).

In [15]:
import re

text = "Order number 12345"
numbers = re.findall(r"\d+", text)
print(numbers)

['12345']


In [12]:
import re

text = "Order number Starting with 12345"
numbers = re.findall(r"\b[A-Z][a-zA-Z]*\b", text)
print(numbers)

['Order', 'Starting']


### 4.3 Python Regex Methods (`search`, `match`, `findall`, `sub`, `split`)
* `re.search(pattern, string)`: Searches **anywhere** in the string for the first match.
* `re.match(pattern, string)`: Checks **only at the beginning** of the string.
* `re.findall(pattern, string)`: Returns a list of **all** non-overlapping matches.
* `re.sub(pattern, repl, string)`: Replaces all matches of `pattern` with `repl`.
* `re.split(pattern, string)`: Splits text wherever `pattern` matches (here `[,\s]+` matches one or more commas or whitespace characters).

In [16]:
# 1. re.search() - searches anywhere
print("search:", re.search(r"pay", "I made a payment"))

# 2. re.match() - checks only from the beginning
print("match at start:", re.match(r"pay", "payment failed"))
print("match in middle:", re.match(r"pay", "I made a payment"))

# 3. re.findall() - returns all matches
print("findall:", re.findall(r"\d+", "Order 123 and invoice 456"))

# 4. re.sub() - replaces matched patterns
print("sub:", re.sub(r"\d+", "", "Order 123"))

# 5. re.split() - splits text on commas and/or whitespace
print("split:", re.split(r"[,\s]+", "apple, banana orange"))

search: <re.Match object; span=(9, 12), match='pay'>
match at start: <re.Match object; span=(0, 3), match='pay'>
match in middle: None
findall: ['123', '456']
sub: Order 
split: ['apple', 'banana', 'orange']


In [13]:
print("split:", re.split(r"[,]+", "apple, banana orange"))

split: ['apple', ' banana orange']


### 4.4 Practical Regex Examples
1. **Extract Email Addresses (`r"[\w.-]+@[\w.-]+\.\w+"`)**:
   * `[\w.-]+`: One or more word characters (`a-z`, `A-Z`, `0-9`, `_`), dots `.`, or hyphens `-`.
   * `@`: Literal `@`.
   * `\.`: Literal dot `.` before the domain extension.
   * `\w+`: One or more word characters for the top-level domain (e.g., `com`).
2. **Extract 10-Digit Phone-Like Numbers (`r"\b\d{10}\b"`)**:
   * `\b`: **Word boundary** — ensures the 10 digits are standalone and not part of an 11- or 12-digit number.
   * `\d{10}`: Exactly 10 consecutive digits.
3. **Extract Hashtags (`r"#\w+"`)**:
   * `#`: Literal hash symbol followed by `\w+` (one or more word characters).
4. **Extract Currency Values (`r"(?:Rs\.?|₹|\$)\s?\d+(?:\.\d{1,2})?"`)**:
   * `(?:...)`: **Non-capturing group** (groups alternatives without creating separate tuple elements in `re.findall()`).
   * `Rs\.?|₹|\$`: Matches `Rs`, `Rs.`, `₹`, or `$` (`\$` escapes the dollar sign).
   * `\s?`: Optional single whitespace character.
   * `\d+`: One or more digits for the whole amount.
   * `(?:\.\d{1,2})?`: Optional decimal part matching a literal dot followed by 1 or 2 digits.
5. **Replace Multiple Spaces (`r"\s+"`)**:
   * Collapses one or more consecutive whitespace characters (`\s+`) into a single space `" "`.

In [17]:
# 1. Extract email addresses
text_email = "Contact abc@gmail.com or xyz@yahoo.com"
emails = re.findall(r"[\w.-]+@[\w.-]+\.\w+", text_email)
print("Emails:", emails)

# 2. Extract phone-like numbers
text_phone = "Call 9876543210"
numbers = re.findall(r"\b\d{10}\b", text_phone)
print("Phone:", numbers)

# 3. Extract hashtags
text_tags = "I love #Python and #DataScience"
hashtags = re.findall(r"#\w+", text_tags)
print("Hashtags:", hashtags)

# 4. Extract currency values
text_curr = "Paid Rs. 500 and $25.50"
values = re.findall(r"(?:Rs\.?|₹|\$)\s?\d+(?:\.\d{1,2})?", text_curr)
print("Currency:", values)

# 5. Replace multiple spaces
text_space = "Payment     failed"
clean_space = re.sub(r"\s+", " ", text_space)
print("Cleaned spaces:", clean_space)

Emails: ['abc@gmail.com', 'xyz@yahoo.com']
Phone: ['9876543210']
Hashtags: ['#Python', '#DataScience']
Currency: ['Rs. 500', '$25.50']
Cleaned spaces: Payment failed


### 4.5 Regex Dry Run
Demonstrates how `r"[^a-zA-Z0-9\s]"` strips `.` and `!!!` from `"Rs. 500!!!"` while preserving letters, spaces, and digits (`500`).

In [18]:
text = "Rs. 500!!!"
matched = re.findall(r"[^a-zA-Z0-9\s]", text)
result = re.sub(r"[^a-zA-Z0-9\s]", "", text)
print("Matched non-alphanumeric chars:", "".join(matched))
print("After replacement:", result)

Matched non-alphanumeric chars: .!!!
After replacement: Rs 500


---
## MODULE 5: TOKENIZATION
### 5.2 & 5.3 Sentence, Word, and Character Tokenization using NLTK
* `sent_tokenize`: Splits text into individual sentences.
* `word_tokenize`: Splits text into words and punctuation tokens based on Penn Treebank rules.
* `list(word)`: Splits a string into individual character tokens.

In [14]:
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import word_tokenize, sent_tokenize

# Word tokenization
text = "I love Data Science"
tokens = word_tokenize(text)
print("Word tokens:", tokens)

# Sentence tokenization
para = "Data Science is interesting. NLP is useful. This is it"
print("Sentence tokens:", sent_tokenize(para))

# Character tokenization
print("Character tokens:", list("cat"))

Word tokens: ['I', 'love', 'Data', 'Science']
Sentence tokens: ['Data Science is interesting.', 'NLP is useful.', 'This is it']
Character tokens: ['c', 'a', 't']


### 5.5 Handling Punctuation and Contractions in Tokenization
NLTK's `word_tokenize` splits contractions like `"can't"` into `['ca', "n't"]` and separates trailing punctuation (`'!'`).

In [20]:
text = "I can't pay online!"
print(word_tokenize(text))

['I', 'ca', "n't", 'pay', 'online', '!']


---
## MODULE 6: STOP WORDS
### 6.3 NLTK Stop Word Removal
Loads standard English stop words and filters out common low-information words while preserving content words.

In [21]:
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words("english"))
print("Is 'the' in stop_words?:", "the" in stop_words)

text = "The customer is requesting a refund"
tokens = word_tokenize(text)
filtered_tokens = [
    word for word in tokens
    if word.lower() not in stop_words
]
print("Filtered tokens:", filtered_tokens)

Is 'the' in stop_words?: True
Filtered tokens: ['customer', 'requesting', 'refund']


---
## MODULE 7: STEMMING
### 7.3 Porter Stemmer & 7.4 Snowball Stemmer
Rule-based algorithms that strip suffixes to reduce words to a root-like stem (which may not always be a valid dictionary word, e.g., `"studies"` → `"studi"`).

In [18]:
from nltk.stem import PorterStemmer, SnowballStemmer

# Porter Stemmer
porter = PorterStemmer()
print("Porter ('playing'):", porter.stem("playing"))
print("Porter ('studies'):", porter.stem("studies"))
print("Porter ('connection'):", porter.stem("connection"))

# Snowball Stemmer
snowball = SnowballStemmer("english")
print("Snowball ('playing'):", snowball.stem("playing"))
print("Snowball ('studies'):", snowball.stem("studies"))

Porter ('playing'): play
Porter ('studies'): studi
Porter ('connection'): connect
Snowball ('playing'): play
Snowball ('studies'): studi


In [20]:
from nltk.stem import SnowballStemmer

snowball = SnowballStemmer("french")
print("French ('jouant'):", snowball.stem("jouant"))

snowball = SnowballStemmer("german")
print("German ('spielen'):", snowball.stem("spielen"))

snowball = SnowballStemmer("spanish")
print("Spanish ('jugando'):", snowball.stem("jugando"))

French ('jouant'): jou
German ('spielen'): spiel
Spanish ('jugando'): jug


---
## MODULE 8: LEMMATIZATION
### 8.2 & 8.3 WordNet Lemmatizer (Default vs. With POS Tags)
Converts words to their valid dictionary base form (lemma). Supplying the Part-of-Speech (`pos="v"` for verb, `pos="a"` for adjective, `pos="n"` for noun, `pos="r"` for adverb) ensures accurate lemmatization.

In [23]:
import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

# Default POS is noun ('n')
print("Default ('studies'):", lemmatizer.lemmatize("studies"))
print("Default ('playing'):", lemmatizer.lemmatize("playing"))

# Specifying POS tag
print("Verb ('playing', pos='v'):", lemmatizer.lemmatize("playing", pos="v"))
print("Adjective ('better', pos='a'):", lemmatizer.lemmatize("better", pos="a"))
print("Verb ('are', pos='v'):", lemmatizer.lemmatize("are", pos="v"))

Default ('studies'): study
Default ('playing'): playing
Verb ('playing', pos='v'): play
Adjective ('better', pos='a'): good
Verb ('are', pos='v'): be


---
## MODULE 10: PART OF SPEECH (POS) TAGGING
### 10.3 NLTK POS Tagging
Assigns grammatical tags (such as `DT` determiner, `JJ` adjective, `NN` noun, `VBZ`/`VBG` verbs, `NNP` proper noun, `RB` adverb) to each token.

In [24]:
import nltk
nltk.download("averaged_perceptron_tagger_eng", quiet=True)
from nltk.tokenize import word_tokenize

text = "The intelligent student is learning Python quickly."
tokens = word_tokenize(text)
tags = nltk.pos_tag(tokens)
print(tags)

[('The', 'DT'), ('intelligent', 'JJ'), ('student', 'NN'), ('is', 'VBZ'), ('learning', 'VBG'), ('Python', 'NNP'), ('quickly', 'RB'), ('.', '.')]


---
## MODULE 11: N-GRAMS
### 11.3 `CountVectorizer` with Bigrams (`ngram_range=(1, 2)`)
Captures both single words (unigrams) and two-word consecutive phrases (bigrams) such as `"not good"`.

In [25]:
from sklearn.feature_extraction.text import CountVectorizer

texts = [
    "not good",
    "good service"
]
vectorizer = CountVectorizer(ngram_range=(1, 2))
X = vectorizer.fit_transform(texts)
print(vectorizer.get_feature_names_out())

['good' 'good service' 'not' 'not good' 'service']


---
## MODULE 12: BAG OF WORDS (BOW)
### 12.3 & 12.4 `CountVectorizer` — Vocabulary Learning & Transforming New Text
* `fit_transform()` learns the vocabulary and builds the document-term frequency matrix.
* `transform()` maps new text onto the already learned vocabulary (unseen words are ignored).

In [26]:
from sklearn.feature_extraction.text import CountVectorizer

documents = [
    "I love data science",
    "I love machine learning",
    "Data science is useful"
]
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(documents)
print("Vocabulary:", vectorizer.get_feature_names_out())
print("Document-Term Matrix:\n", X.toarray())

# Transform new text using existing vocabulary
new_text = ["I love data"]
new_vector = vectorizer.transform(new_text)
print("New text vector:\n", new_vector.toarray())

Vocabulary: ['data' 'is' 'learning' 'love' 'machine' 'science' 'useful']
Document-Term Matrix:
 [[1 0 0 1 0 1 0]
 [0 0 1 1 1 0 0]
 [1 1 0 0 0 1 1]]
New text vector:
 [[1 0 0 1 0 0 0]]


---
## MODULE 13: TF-IDF
### 13.7 & 13.8 `TfidfVectorizer` and Important Hyperparameters
Weights terms by multiplying Term Frequency (TF) by Inverse Document Frequency (IDF), emphasizing distinctive words over common words.

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer

documents = [
    "fraud payment fraud",
    "payment failed",
    "fraud detected"
]
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(documents)
print("Vocabulary:", vectorizer.get_feature_names_out())
print("TF-IDF Matrix:\n", X.toarray())

# Configured TfidfVectorizer with common production parameters
tuned_vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True
)
print("\nConfigured TfidfVectorizer:", tuned_vectorizer)

Vocabulary: ['detected' 'failed' 'fraud' 'payment']
TF-IDF Matrix:
 [[0.         0.         0.89442719 0.4472136 ]
 [0.         0.79596054 0.         0.60534851]
 [0.79596054 0.         0.60534851 0.        ]]

Configured TfidfVectorizer: TfidfVectorizer(max_df=0.95, max_features=5000, min_df=2, ngram_range=(1, 2),
                sublinear_tf=True)


---
## MODULE 15: TEXT FEATURE ENGINEERING
### 15.2 & 15.3 Creating Numerical & Keyword Features using `pandas` and `re`
**Regex Explanations used below:**
* `r"https?://\S+|www\.\S+"`: Counts URLs starting with `http://`, `https://`, or `www.` followed by non-whitespace (`\S+`).
* `r"\?"`: Escapes `?` to count literal question marks.
* `r"\bfraud\b|\bunauthorized\b|\bstolen\b"`: Uses word boundaries (`\b`) around `fraud`, `unauthorized`, and `stolen` so partial word matches are avoided.

In [ ]:
import pandas as pd
import re

df = pd.DataFrame({
    "text": [
        "Payment failed!!!",
        "I love this product 😊",
        "Visit www.example.com"
    ]
})

df["text"] = df["text"].fillna("")
df["character_count"] = df["text"].str.len()
df["word_count"] = df["text"].str.split().str.len()
df["uppercase_count"] = df["text"].apply(
    lambda x: sum(char.isupper() for char in x)
)
df["digit_count"] = df["text"].apply(
    lambda x: sum(char.isdigit() for char in x)
)
df["punctuation_count"] = df["text"].apply(
    lambda x: sum(not char.isalnum() and not char.isspace()
                  for char in x)
)
df["url_count"] = df["text"].apply(
    lambda x: len(re.findall(r"https?://\S+|www\.\S+", x))
)
df["exclamation_count"] = df["text"].str.count("!")
df["question_count"] = df["text"].str.count(r"\?")
df["unique_word_count"] = df["text"].apply(
    lambda x: len(set(x.lower().split()))
) 

# Domain keyword feature
df["fraud_keyword_count"] = df["text"].str.lower().str.count(
    r"\bfraud\b|\bunauthorized\b|\bstolen\b"
)

print(df)

                    text  character_count  word_count  uppercase_count  \
0      Payment failed!!!               17           2                1   
1  I love this product 😊               21           5                1   
2  Visit www.example.com               21           2                1   

   digit_count  punctuation_count  url_count  exclamation_count  \
0            0                  3          0                  3   
1            0                  1          0                  0   
2            0                  2          1                  0   

   question_count  unique_word_count  fraud_keyword_count  
0               0                  2                    0  
1               0                  5                    0  
2               0                  2                    0  


In [29]:
df

,text,character_count,word_count,uppercase_count,digit_count,punctuation_count,url_count,exclamation_count,question_count,unique_word_count,fraud_keyword_count
0,Payment failed!!!,17,2,1,0,3,0,3,0,2,0
1,I love this product 😊,21,5,1,0,1,0,0,0,5,0
2,Visit www.example.com,21,2,1,0,2,1,0,0,2,0


---
## MODULE 18: NLP DATA LEAKAGE AND BEST PRACTICES
### 18.2, 18.5 & 18.6 Preventing Vocabulary Leakage with `train_test_split` and `Pipeline`
Always split text into training and test sets **before** fitting `TfidfVectorizer`, or wrap both steps inside a scikit-learn `Pipeline` so test data never influences vocabulary or IDF calculations.

In [31]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Sample dataset
text = [
    "Unauthorized transaction on my account",
    "Someone stole my card details",
    "Payment failed during checkout",
    "Bill payment is still pending",
    "Loan EMI amount is wrong",
    "Need loan foreclosure details",
    "Cannot login to mobile app",
    "Account locked after wrong password",
    "Unauthorized debit alert",
    "Transfer failed money deducted"
]
y = [
    "Fraud", "Fraud", "Payment", "Payment", "Loan",
    "Loan", "Access", "Access", "Fraud", "Payment"
]

# Split BEFORE vectorizing
X_train_text, X_test_text, y_train, y_test = train_test_split(
    text,
    y,
    test_size=0.4,
    random_state=42,
    stratify=y
)

# Method 1: Manual fit on train, transform on test
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

# Method 2: Using a scikit-learn Pipeline (recommended)
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("model", LogisticRegression())
])

pipeline.fit(X_train_text, y_train)
y_pred = pipeline.predict(X_test_text)
print("Predictions:", y_pred)

Predictions: ['Payment' 'Fraud' 'Payment' 'Payment']


---
## MODULE 19: END-TO-END NLP PROJECT — Customer Complaint Classification System
A complete, runnable multiclass classification pipeline predicting 6 banking complaint categories (`Fraud`, `Payment Issue`, `Loan Issue`, `Credit Card Issue`, `Account Access`, `Customer Service`).

### STEP 1: Import Libraries

In [32]:
import pandas as pd
import numpy as np
import random
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

### STEP 2 & STEP 3: Generate Synthetic Dataset (3,000 Records) & Display First Five Rows
Generates realistic multi-column banking complaints with occasional misspellings, informal slang, and 3% missing values across text fields.

In [33]:
random.seed(42)
np.random.seed(42)

categories = [
    "Fraud",
    "Payment Issue",
    "Loan Issue",
    "Credit Card Issue",
    "Account Access",
    "Customer Service"
]

complaint_templates = {
    "Fraud": [
        "I noticed an unauthorized transaction on my account",
        "Someone used my card without my permission",
        "Please block my account because I suspect fraud",
        "I did not make this payment and need an investigation",
        "Unknown transaction deducted money from my account",
        "My account was used by someone else",
        "There is a suspicious charge on my statement"
    ],
    "Payment Issue": [
        "My payment failed but money was deducted",
        "The payment is still pending after several hours",
        "I paid the bill but the amount is not updated",
        "The transaction failed during checkout",
        "I made the payment twice by mistake",
        "Money was deducted but the merchant did not receive it",
        "My online transfer is not completed"
    ],
    "Loan Issue": [
        "I want to know the status of my loan application",
        "My loan EMI amount is incorrect",
        "Please explain the interest charged on my loan",
        "My loan repayment is not reflected",
        "I need help with loan foreclosure",
        "The loan approval is taking too long",
        "I want to update my loan repayment schedule"
    ],
    "Credit Card Issue": [
        "My credit card is blocked and I cannot use it",
        "The credit card limit is not updated",
        "I was charged an incorrect annual fee",
        "My credit card statement has an error",
        "I need to dispute a credit card charge",
        "My card payment due date is incorrect",
        "The credit card is not working at the merchant"
    ],
    "Account Access": [
        "I cannot log in to my banking account",
        "My password reset link is not working",
        "I am unable to access mobile banking",
        "My account is locked after multiple attempts",
        "The OTP is not arriving on my phone",
        "I forgot my login credentials",
        "The application keeps showing login error"
    ],
    "Customer Service": [
        "I have been waiting for customer support",
        "The agent did not resolve my complaint",
        "Please call me regarding my service request",
        "I received incorrect information from support",
        "My complaint has been open for several days",
        "The customer service experience was poor",
        "I need assistance with my service request"
    ]
}

titles = {
    "Fraud": [
        "Unauthorized transaction",
        "Suspicious payment",
        "Possible account fraud"
    ],
    "Payment Issue": [
        "Payment failed",
        "Payment pending",
        "Amount not updated"
    ],
    "Loan Issue": [
        "Loan application status",
        "Incorrect EMI",
        "Loan repayment issue"
    ],
    "Credit Card Issue": [
        "Credit card blocked",
        "Incorrect card fee",
        "Card limit issue"
    ],
    "Account Access": [
        "Login problem",
        "Account locked",
        "OTP not received"
    ],
    "Customer Service": [
        "Poor customer support",
        "Unresolved complaint",
        "Need service assistance"
    ]
}

agent_notes = {
    "Fraud": [
        "Customer reported suspicious card usage",
        "Account security investigation required",
        "Potential unauthorized transaction reported"
    ],
    "Payment Issue": [
        "Payment status requires verification",
        "Transaction reconciliation pending",
        "Payment complaint received"
    ],
    "Loan Issue": [
        "Loan servicing team needs to review",
        "Loan account query received",
        "Repayment details require verification"
    ],
    "Credit Card Issue": [
        "Credit card servicing request received",
        "Card account requires investigation",
        "Credit card complaint registered"
    ],
    "Account Access": [
        "Customer unable to authenticate",
        "Login support required",
        "Account access issue reported"
    ],
    "Customer Service": [
        "Customer requested support follow-up",
        "Service quality complaint registered",
        "Customer requires further assistance"
    ]
}

records = []
for i in range(3000):
    category = random.choice(categories)
    description = random.choice(
        complaint_templates[category]
    )
    title = random.choice(titles[category])
    message = random.choice([
        description,
        description + " Please help me urgently.",
        description + " Kindly resolve this asap.",
        description + " This is very frustrating.",
        description + " Pls check and update me.",
        description + " I need a quick resolution."
    ])
    # Add occasional spelling mistakes
    if random.random() < 0.08:
        message = message.replace("payment", "paymnt")
        message = message.replace("account", "acount")
    # Add informal language
    if random.random() < 0.10:
        message += " pls help"
    previous_text = random.choice([
        "",
        "I contacted support earlier",
        "This issue happened last month",
        "I have already raised a request"
    ])
    records.append({
        "complaint_id": f"C{i+1:05d}",
        "customer_age": random.randint(18, 75),
        "customer_tenure_months": random.randint(1, 180),
        "product_type": random.choice([
            "Savings Account",
            "Credit Card",
            "Personal Loan",
            "Mobile Banking"
        ]),
        "complaint_channel": random.choice([
            "Email",
            "Phone",
            "Mobile App",
            "Website"
        ]),
        "transaction_amount": round(
            random.uniform(100, 50000), 2
        ),
        "customer_segment": random.choice([
            "Retail",
            "Premium",
            "Business"
        ]),
        "complaint_title": title,
        "complaint_description": description,
        "customer_message": message,
        "agent_notes": random.choice(
            agent_notes[category]
        ),
        "previous_complaint_text": previous_text,
        "complaint_category": category
    })

df = pd.DataFrame(records)

# Introduce missing text values
for column in [
    "complaint_title",
    "complaint_description",
    "customer_message",
    "agent_notes",
    "previous_complaint_text"
]:
    df.loc[
        df.sample(frac=0.03, random_state=42).index,
        column
    ] = np.nan

print("Dataset Shape:", df.shape)
print(df.head())

Dataset Shape: (3000, 13)
  complaint_id  customer_age  customer_tenure_months     product_type  \
0       C00001            61                     140  Savings Account   
1       C00002            32                     115    Personal Loan   
2       C00003            23                      98  Savings Account   
3       C00004            53                      76    Personal Loan   
4       C00005            35                     117    Personal Loan   

  complaint_channel  transaction_amount customer_segment  \
0           Website             1685.96           Retail   
1             Email            37964.49           Retail   
2        Mobile App            42389.97         Business   
3             Phone            35258.13           Retail   
4             Phone            18572.03           Retail   

         complaint_title                     complaint_description  \
0  Poor customer support  I have been waiting for customer support   
1     Amount not updated       I m

In [34]:
df

,complaint_id,customer_age,customer_tenure_months,product_type,complaint_channel,transaction_amount,customer_segment,complaint_title,complaint_description,customer_message,agent_notes,previous_complaint_text,complaint_category
0,C00001,61,140,Savings Account,Website,1685.96,Retail,Poor customer support,I have been waiting for customer support,I have been waiting for customer support I nee...,Customer requested support follow-up,,Customer Service
1,C00002,32,115,Personal Loan,Email,37964.49,Retail,Amount not updated,I made the payment twice by mistake,I made the payment twice by mistake,Payment complaint received,I have already raised a request,Payment Issue
2,C00003,23,98,Savings Account,Mobile App,42389.97,Business,Incorrect card fee,I was charged an incorrect annual fee,I was charged an incorrect annual fee Please h...,Card account requires investigation,,Credit Card Issue
3,C00004,53,76,Personal Loan,Phone,35258.13,Retail,Suspicious payment,My account was used by someone else,My account was used by someone else Pls check ...,Potential unauthorized transaction reported,,Fraud
4,C00005,35,117,Personal Loan,Phone,18572.03,Retail,Payment pending,My online transfer is not completed,My online transfer is not completed,Payment complaint received,I have already raised a request,Payment Issue
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,C02996,55,56,Credit Card,Mobile App,9043.68,Business,Credit card blocked,My credit card is blocked and I cannot use it,My credit card is blocked and I cannot use it ...,Credit card servicing request received,I contacted support earlier,Credit Card Issue
2996,C02997,71,57,Credit Card,Email,21889.54,Premium,Payment pending,My payment failed but money was deducted,My payment failed but money was deducted pls help,Payment complaint received,This issue happened last month,Payment Issue
2997,C02998,69,115,Mobile Banking,Phone,10839.40,Business,Need service assistance,My complaint has been open for several days,My complaint has been open for several days I ...,Customer requested support follow-up,This issue happened last month,Customer Service
2998,C02999,27,114,Personal Loan,Phone,28411.23,Premium,Amount not updated,I made the payment twice by mistake,I made the payment twice by mistake,Transaction reconciliation pending,,Payment Issue


### STEP 4: Exploratory Data Analysis (EDA)
Inspects shape, data types, missing values, duplicates, class balance, and message length statistics.

In [35]:
# Shape
print("Shape:", df.shape)

# Data types
print("\nData Types:\n", df.dtypes)

# Missing values
print("\nMissing Values:\n", df.isnull().sum())

# Duplicate records
print("\nDuplicate rows:", df.duplicated().sum())

# Target distribution
print("\nTarget Counts:\n", df["complaint_category"].value_counts())

# Target distribution percentage
print(
    "\nTarget Percentage:\n",
    df["complaint_category"].value_counts(normalize=True) * 100
)

# Text length distribution
df["message_length"] = (
    df["customer_message"].fillna("").str.len()
)
print("\nMessage Length Stats:\n", df["message_length"].describe())

# Most frequent complaint categories
print("\nTop 6 Categories:\n", df["complaint_category"].value_counts().head(6))

Shape: (3000, 13)

Data Types:
 complaint_id                   str
customer_age                 int64
customer_tenure_months       int64
product_type                   str
complaint_channel              str
transaction_amount         float64
customer_segment               str
complaint_title                str
complaint_description          str
customer_message               str
agent_notes                    str
previous_complaint_text        str
complaint_category             str
dtype: object

Missing Values:
 complaint_id                0
customer_age                0
customer_tenure_months      0
product_type                0
complaint_channel           0
transaction_amount          0
customer_segment            0
complaint_title            90
complaint_description      90
customer_message           90
agent_notes                90
previous_complaint_text    90
complaint_category          0
dtype: int64

Duplicate rows: 0

Target Counts:
 complaint_category
Customer Service     51

### STEP 5, STEP 6 & STEP 7: Handle Missing Values, Clean Text & Combine Text Columns
**Regex Explanations in `clean_text()`:**
* `r"<.*?>"` → Non-greedy match for HTML tags (`<...>`).
* `r"https?://\S+|www\.\S+"` → Matches `http://`, `https://`, or `www.` URLs up to the next whitespace (`\S+`).
* `r"\s+"` → Collapses one or more consecutive whitespace characters (`\s+`) into a single space `" "`.

In [36]:
# STEP 5: Handle missing values
text_columns = [
    "complaint_title",
    "complaint_description",
    "customer_message",
    "agent_notes",
    "previous_complaint_text"
]
df[text_columns] = df[text_columns].fillna("")

# STEP 6: Text preprocessing
def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text
    )
    text = re.sub(r"\s+", " ", text)
    return text.strip()

for column in text_columns:
    df[column] = df[column].apply(clean_text)

# STEP 7: Combine text columns
df["combined_text"] = (
    df["complaint_title"] + " " +
    df["complaint_description"] + " " +
    df["customer_message"] + " " +
    df["agent_notes"] + " " +
    df["previous_complaint_text"]
)

print("Sample combined text:\n", df["combined_text"].iloc[0])

Sample combined text:
 poor customer support i have been waiting for customer support i have been waiting for customer support i need a quick resolution. customer requested support follow-up 


### STEP 8 & STEP 9: Stratified Train-Test Split & TF-IDF Vectorization
Splits data 80/20 with stratification, then fits `TfidfVectorizer` (unigrams + bigrams, sublinear TF) strictly on `X_train`.

In [37]:
# STEP 8: Train-test split
X = df["combined_text"]
y = df["complaint_category"]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# STEP 9: TF-IDF vectorization
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("First 20 vocabulary terms:", vectorizer.get_feature_names_out()[:20])
print("X_train_tfidf shape:", X_train_tfidf.shape)

First 20 vocabulary terms: ['access' 'access issue' 'access mobile' 'account' 'account access'
 'account account' 'account because' 'account cannot' 'account customer'
 'account fraud' 'account is' 'account kindly' 'account locked'
 'account need' 'account noticed' 'account please' 'account pls'
 'account potential' 'account query' 'account requires']
X_train_tfidf shape: (2400, 966)


### STEP 10 & STEP 11: Train & Evaluate ML Models (Naive Bayes, Logistic Regression, Linear SVM)
Trains three classifiers and compares Accuracy, Weighted Precision, Weighted Recall, and Weighted F1-score, plus a detailed classification report and confusion matrix.

In [38]:
# STEP 10: Train Machine Learning models
# Model 1: Multinomial Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_pred = nb_model.predict(X_test_tfidf)

# Model 2: Logistic Regression
lr_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)
lr_model.fit(X_train_tfidf, y_train)
lr_pred = lr_model.predict(X_test_tfidf)

# Model 3: Linear SVM
svm_model = LinearSVC(
    class_weight="balanced"
)
svm_model.fit(X_train_tfidf, y_train)
svm_pred = svm_model.predict(X_test_tfidf)

# STEP 11: Evaluate models
results = []
for model_name, predictions in [
    ("Naive Bayes", nb_pred),
    ("Logistic Regression", lr_pred),
    ("Linear SVM", svm_pred)
]:
    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(
            y_test, predictions
        ),
        "Precision": precision_score(
            y_test, predictions,
            average="weighted",
            zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions,
            average="weighted",
            zero_division=0
        ),
        "F1": f1_score(
            y_test, predictions,
            average="weighted",
            zero_division=0
        )
    })

results_df = pd.DataFrame(results)
print("Model Comparison:\n", results_df.sort_values("F1", ascending=False))

print("\nLogistic Regression Classification Report:\n", classification_report(
    y_test,
    lr_pred,
    zero_division=0
))

print("Logistic Regression Confusion Matrix:\n", confusion_matrix(
    y_test,
    lr_pred
))

Model Comparison:
                  Model  Accuracy  Precision    Recall        F1
0          Naive Bayes  0.981667   0.983436  0.981667  0.981982
2           Linear SVM  0.980000   0.982143  0.980000  0.980381
1  Logistic Regression  0.973333   0.976980  0.973333  0.974059

Logistic Regression Classification Report:
                    precision    recall  f1-score   support

   Account Access       1.00      0.99      0.99        97
Credit Card Issue       1.00      0.96      0.98       100
 Customer Service       1.00      0.95      0.98       103
            Fraud       0.86      1.00      0.93       101
       Loan Issue       1.00      0.97      0.98        97
    Payment Issue       1.00      0.97      0.99       102

         accuracy                           0.97       600
        macro avg       0.98      0.97      0.97       600
     weighted avg       0.98      0.97      0.97       600

Logistic Regression Confusion Matrix:
 [[ 96   0   0   1   0   0]
 [  0  96   0   4   0

### STEP 13, STEP 14 & STEP 15: Predict New Complaints, Class Probabilities & Manual Threshold Tuning
Uses the trained `LogisticRegression` model to predict categories and class probabilities for unseen complaints, flags high-risk fraud alerts, and demonstrates binary threshold tuning (`0.50`, `0.30`, `0.80`).

In [40]:
# STEP 13: Predict new complaints
new_complaints = [
    "Someone used my card without permission",
    "My payment failed but money was deducted",
    "I cannot log in to my mobile banking account",
    "My loan EMI amount is incorrect"
]
new_tfidf = vectorizer.transform(new_complaints)
new_predictions = lr_model.predict(new_tfidf)
new_probabilities = lr_model.predict_proba(new_tfidf)
print("Predicted Categories:", new_predictions)

# Prediction probability for each class
probability_df = pd.DataFrame(
    new_probabilities,
    columns=lr_model.classes_
)
print("\nClass Probabilities:\n", probability_df.round(4))

# STEP 14: Create prediction DataFrame
prediction_df = pd.DataFrame({
    "complaint_text": new_complaints,
    "predicted_category": new_predictions
})
prediction_df["fraud_probability"] = (
    probability_df["Fraud"].values
)
prediction_df["prediction_status"] = np.where(
    prediction_df["fraud_probability"] >= 0.50,
    "Fraud Alert",
    "Normal"
)
print("\nFinal Prediction Summary:\n", prediction_df)

# STEP 15: Manual threshold tuning (Binary Fraud Example)
fraud_probability = np.array([
    0.15, 0.35, 0.55, 0.80, 0.92
])

for thresh in [0.50, 0.30, 0.80]:
    fraud_prediction = (fraud_probability >= thresh).astype(int)
    print(f"Threshold {thresh:.2f} -> Predictions: {fraud_prediction}")

Predicted Categories: ['Fraud' 'Payment Issue' 'Account Access' 'Loan Issue']

Class Probabilities:
    Account Access  Credit Card Issue  Customer Service   Fraud  Loan Issue  \
0          0.0253             0.0711            0.0245  0.8190      0.0315   
1          0.0087             0.0118            0.0085  0.0182      0.0099   
2          0.8826             0.0275            0.0163  0.0274      0.0258   
3          0.0124             0.0242            0.0123  0.0151      0.9127   

   Payment Issue  
0         0.0286  
1         0.9429  
2         0.0204  
3         0.0232  

Final Prediction Summary:
                                  complaint_text predicted_category  \
0       Someone used my card without permission              Fraud   
1      My payment failed but money was deducted      Payment Issue   
2  I cannot log in to my mobile banking account     Account Access   
3               My loan EMI amount is incorrect         Loan Issue   

   fraud_probability prediction_st

---
## MODULE 20: SENTIMENT ANALYSIS — BRIEF INTRODUCTION
### 20.3 VADER Sentiment Analysis
Uses NLTK's `SentimentIntensityAnalyzer` (VADER) to compute `neg`, `neu`, `pos`, and `compound` sentiment polarity scores (`-1` to `+1`).

In [22]:
import nltk
nltk.download("vader_lexicon", quiet=True)
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()
text = "The product is excellent, but delivery was very slow."
scores = sia.polarity_scores(text)
print(scores)

{'neg': 0.0, 'neu': 0.773, 'pos': 0.227, 'compound': 0.3291}


In [42]:
text = "The screen is working sluggy."
scores = sia.polarity_scores(text)
print(scores)

{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}


In [23]:
text = "The screen is very dark and not watchable for movies"
scores = sia.polarity_scores(text)
print(scores)

{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}


In [24]:
text = "The screen is bad"
scores = sia.polarity_scores(text)
print(scores)

{'neg': 0.538, 'neu': 0.462, 'pos': 0.0, 'compound': -0.5423}


In [25]:
text = "The screen is not good and very bad battery, do not purchase"
scores = sia.polarity_scores(text)
print(scores)

{'neg': 0.383, 'neu': 0.617, 'pos': 0.0, 'compound': -0.7351}


In [26]:
text = "this is very good thing"
scores = sia.polarity_scores(text)
print(scores)

{'neg': 0.0, 'neu': 0.556, 'pos': 0.444, 'compound': 0.4927}
